# Test `data_access.py` tools

Notebook này dùng để test từng method trong `ShoppingDataStore` và từng LangChain tool được tạo bởi `build_data_tools()`.

Chạy notebook từ root project: `Day09-MultiAgent-Architecture`.

In [42]:
from pathlib import Path
import json
import sys
from pprint import pprint

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from app.data_access import ShoppingDataStore, build_data_tools

DATA_PATH = ROOT / "data" / "order_customer_mock_data.json"
store = ShoppingDataStore(DATA_PATH)

print("Root:", ROOT)
print("Data path exists:", DATA_PATH.exists())
print("Customers:", len(store.customers))
print("Orders:", len(store.orders))
print("Vouchers:", len(store.vouchers))
print("Dataset today:", store.today)

Root: /Users/thangnguyen/Documents/VinAI - Lab/Day09-MultiAgent-Architecture
Data path exists: True
Customers: 80
Orders: 360
Vouchers: 284
Dataset today: 2026-06-07


## 1. Test `get_customer_by_id`

In [43]:
result = store.get_customer_by_id("C002")
assert result["status"] == "ok"
pprint(result)

not_found = store.get_customer_by_id("C999")
assert not_found["status"] == "not_found"
pprint(not_found)

{'customer': {'account_status': 'active',
              'city': 'Can Tho',
              'customer_id': 'C002',
              'customer_name': 'Đỗ Gia Uyên',
              'default_address': {'city': 'Can Tho',
                                  'country': 'VN',
                                  'district': 'Ninh Kieu',
                                  'line1': '81 Duong Dien Bien Phu'},
              'district': 'Ninh Kieu',
              'email': 'dỗ.gia.uyen02@example.com',
              'gender': 'female',
              'joined_at': '2024-01-18T15:30',
              'latest_order_id': '2192',
              'loyalty_points': 980,
              'max_voucher_per_month': 6,
              'phone': '0984025970',
              'preferred_categories': ['beauty', 'fashion', 'home'],
              'remaining_voucher_quota_this_month': 5,
              'support_flags': ['high_value_customer', 'none'],
              'tier': 'Standard',
              'total_orders': 5,
              'total_spen

## 2. Test `get_orders_by_customer_id`

In [44]:
result = store.get_orders_by_customer_id("C001", limit=3)
assert result["status"] == "ok"
assert result["count"] <= 3

summary = [
    {
        "order_id": order["order_id"],
        "created_at": order["created_at"],
        "order_status": order["order_status"],
        "estimated_delivery": order.get("estimated_delivery"),
    }
    for order in result["orders"]
]
pprint(summary)

not_found = store.get_orders_by_customer_id("C999")
assert not_found["status"] == "not_found"
pprint(not_found)

[{'created_at': '2026-06-05T10:15',
  'estimated_delivery': '2026-06-09',
  'order_id': '1971',
  'order_status': 'in_transit'},
 {'created_at': '2026-05-16T14:35',
  'estimated_delivery': '2026-05-19',
  'order_id': '1901',
  'order_status': 'completed'},
 {'created_at': '2026-02-21T12:20',
  'estimated_delivery': '2026-02-24',
  'order_id': '2160',
  'order_status': 'in_transit'}]
{'customer_id': 'C999', 'status': 'not_found'}


## 3. Test `get_order_detail_by_order_id`

In [45]:
for order_id in ["1971", "2058"]:
    result = store.get_order_detail_by_order_id(order_id)
    assert result["status"] == "ok"
    order = result["order"]
    print("=" * 80)
    print("order_id:", order["order_id"])
    print("customer_id:", order["customer_id"])
    print("status:", order["order_status"])
    print("estimated_delivery:", order.get("estimated_delivery"))
    print("can_return_now:", order.get("can_return_now"))
    print("eligible_for_return_until:", order.get("eligible_for_return_until"))
    print("latest_status_note:", order.get("latest_status_note"))

not_found = store.get_order_detail_by_order_id("9999")
assert not_found["status"] == "not_found"
pprint(not_found)

order_id: 1971
customer_id: C001
status: in_transit
estimated_delivery: 2026-06-09
can_return_now: False
eligible_for_return_until: None
latest_status_note: Đơn hàng đang trên đường giao, dự kiến giao trong ngày 2026-06-09.
order_id: 2058
customer_id: C014
status: delivered
estimated_delivery: 2026-06-02
can_return_now: True
eligible_for_return_until: 2026-06-16
latest_status_note: Đơn đã giao thành công và vẫn còn trong thời gian hỗ trợ trả hàng.
{'order_id': '9999', 'status': 'not_found'}


## 4. Test `get_vouchers_by_customer_id`

In [46]:
all_vouchers = store.get_vouchers_by_customer_id("C001")
active_vouchers = store.get_vouchers_by_customer_id("C001", only_active=True)

assert all_vouchers["status"] == "ok"
assert active_vouchers["status"] == "ok"
assert active_vouchers["count"] <= all_vouchers["count"]

print("All vouchers:", all_vouchers["count"])
print("Active vouchers:", active_vouchers["count"])
pprint(active_vouchers["vouchers"])

not_found = store.get_vouchers_by_customer_id("C999")
assert not_found["status"] == "not_found"
pprint(not_found)

All vouchers: 5
Active vouchers: 1
[{'assigned_order_id': None,
  'customer_id': 'C001',
  'discount_type': 'fixed',
  'discount_value': 50000,
  'end_at': '2026-06-30T23:59',
  'max_discount': 50000,
  'min_order_value': 399000,
  'remaining_uses': 1,
  'restored_after_cancellation': False,
  'source': 'birthday_reward',
  'stackable_with': ['shop', 'shipping'],
  'start_at': '2026-06-01T00:00',
  'status': 'active',
  'title': 'Voucher sinh nhật khách hàng C001',
  'voucher_code': 'C001-BDAY-0626',
  'voucher_type': 'platform'}]
{'customer_id': 'C999', 'status': 'not_found'}


## 5. Build LangChain tools

In [47]:
tools = build_data_tools(store)
tool_map = {tool.name: tool for tool in tools}

print("Tools:")
for tool in tools:
    print("-", tool.name, "|", tool.description)

expected_tools = {
    "get_customer_by_id",
    "get_orders_by_customer_id",
    "get_order_detail_by_order_id",
    "get_vouchers_by_customer_id",
}
assert set(tool_map) == expected_tools

Tools:
- get_customer_by_id | Tra cứu thông tin khách hàng theo customer_id, ví dụ C001.
- get_orders_by_customer_id | Lấy danh sách đơn hàng gần nhất của một khách hàng theo customer_id.
- get_order_detail_by_order_id | Tra cứu chi tiết một đơn hàng theo order_id, ví dụ 1971 hoặc 2058.
- get_vouchers_by_customer_id | Lấy voucher của khách hàng; đặt only_active=True để chỉ lấy voucher còn dùng được.


## 6. Invoke từng LangChain tool

In [48]:
customer_result = tool_map["get_customer_by_id"].invoke({"customer_id": "C001"})
assert customer_result["status"] == "ok"
pprint(customer_result)

orders_result = tool_map["get_orders_by_customer_id"].invoke({"customer_id": "C001", "limit": 2})
assert orders_result["status"] == "ok"
assert orders_result["count"] <= 2
pprint(orders_result)

order_result = tool_map["get_order_detail_by_order_id"].invoke({"order_id": "1971"})
assert order_result["status"] == "ok"
pprint(order_result)

voucher_result = tool_map["get_vouchers_by_customer_id"].invoke({"customer_id": "C001", "only_active": True})
assert voucher_result["status"] == "ok"
pprint(voucher_result)

{'customer': {'account_status': 'active',
              'city': 'Ho Chi Minh',
              'customer_id': 'C001',
              'customer_name': 'Nguyen Thu Anh',
              'default_address': {'city': 'Ho Chi Minh',
                                  'country': 'VN',
                                  'district': 'Binh Thanh',
                                  'line1': '128 Nguyen Gia Tri'},
              'district': 'Binh Thanh',
              'email': 'nguyen.thu.anh01@example.com',
              'gender': 'female',
              'joined_at': '2024-02-05T14:35',
              'latest_order_id': '2259',
              'loyalty_points': 2416,
              'max_voucher_per_month': 10,
              'phone': '0901234567',
              'preferred_categories': ['fashion', 'beauty', 'home'],
              'remaining_voucher_quota_this_month': 6,
              'support_flags': ['frequent_buyer'],
              'tier': 'Gold',
              'total_orders': 4,
              'total_spent':

## 7. Quick regression tests

Cell này chạy các assertion ngắn cho những case quan trọng trong lab.

In [49]:
assert store.get_customer_by_id("C001")["status"] == "ok"
assert store.get_customer_by_id("C999")["status"] == "not_found"

order_1971 = store.get_order_detail_by_order_id("1971")
assert order_1971["status"] == "ok"
assert order_1971["order"]["estimated_delivery"] == "2026-06-09"

order_2058 = store.get_order_detail_by_order_id("2058")
assert order_2058["status"] == "ok"
assert order_2058["order"]["can_return_now"] is True

assert store.get_order_detail_by_order_id("9999")["status"] == "not_found"
assert store.get_vouchers_by_customer_id("C001", only_active=True)["count"] >= 1

print("All regression tests passed.")

All regression tests passed.


## 8. Test `parse_policy_markdown`

Các cell dưới đây test riêng parser RAG policy dùng `MarkdownHeaderTextSplitter`.

In [50]:
from rag.parser import parse_policy_markdown

POLICY_PATH = ROOT / "data" / "policy_mock_vi.md"
policy_text = POLICY_PATH.read_text(encoding="utf-8")
policy_chunks = parse_policy_markdown(policy_text)

print("Chunks:", len(policy_chunks))
print("Type:", type(policy_chunks[0]).__name__)
print(policy_chunks[0])

Chunks: 38
Type: dict
{'section_h2': '1. Mục đích tài liệu', 'section_h3': None, 'citation': '1. Mục đích tài liệu', 'content': 'Tài liệu này mô tả các quy định mẫu dành cho một nền tảng mua sắm online giả lập tên là **VinShop Demo**. Mục tiêu là tạo ra một nguồn dữ liệu đủ dài, đủ chi tiết và có cấu trúc rõ ràng để:  \n- dùng làm knowledge base cho `Policy / RAG Agent`\n- dùng để trích xuất câu trả lời có citation theo từng mục\n- kết hợp với dữ liệu đơn hàng thực tế để trả lời các câu hỏi hỗn hợp  \nVí dụ:  \n- "Chính sách hoàn trả hàng ra sao?"\n- "Đơn hàng 1971 có được hoàn trả không?"\n- "Voucher có được hoàn lại khi hủy đơn không?"\n- "Giao hàng hỏa tốc mất bao lâu?"', 'rendered_text': '## 1. Mục đích tài liệu\n\nTài liệu này mô tả các quy định mẫu dành cho một nền tảng mua sắm online giả lập tên là **VinShop Demo**. Mục tiêu là tạo ra một nguồn dữ liệu đủ dài, đủ chi tiết và có cấu trúc rõ ràng để:  \n- dùng làm knowledge base cho `Policy / RAG Agent`\n- dùng để trích xuất câu t

## 9. Xem các text chunk mẫu

In [51]:
policy_chunks[:5]

[{'section_h2': '1. Mục đích tài liệu',
  'section_h3': None,
  'citation': '1. Mục đích tài liệu',
  'content': 'Tài liệu này mô tả các quy định mẫu dành cho một nền tảng mua sắm online giả lập tên là **VinShop Demo**. Mục tiêu là tạo ra một nguồn dữ liệu đủ dài, đủ chi tiết và có cấu trúc rõ ràng để:  \n- dùng làm knowledge base cho `Policy / RAG Agent`\n- dùng để trích xuất câu trả lời có citation theo từng mục\n- kết hợp với dữ liệu đơn hàng thực tế để trả lời các câu hỏi hỗn hợp  \nVí dụ:  \n- "Chính sách hoàn trả hàng ra sao?"\n- "Đơn hàng 1971 có được hoàn trả không?"\n- "Voucher có được hoàn lại khi hủy đơn không?"\n- "Giao hàng hỏa tốc mất bao lâu?"',
  'rendered_text': '## 1. Mục đích tài liệu\n\nTài liệu này mô tả các quy định mẫu dành cho một nền tảng mua sắm online giả lập tên là **VinShop Demo**. Mục tiêu là tạo ra một nguồn dữ liệu đủ dài, đủ chi tiết và có cấu trúc rõ ràng để:  \n- dùng làm knowledge base cho `Policy / RAG Agent`\n- dùng để trích xuất câu trả lời có cit

## 10. Regression tests cho markdown chunks

## 11. Test `ChromaPolicyStore` nhanh bằng fake embeddings

Cell này test logic Chroma add/search mà không cần tải model embedding thật.

In [53]:
from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory
import importlib

import rag.parser as parser_module
import rag.vector_store as vector_store_module

# Reload để notebook dùng code mới nhất sau khi bạn sửa file .py.
importlib.reload(parser_module)
importlib.reload(vector_store_module)
from rag.vector_store import ChromaPolicyStore

ROOT = Path.cwd()
POLICY_PATH = ROOT / "data" / "policy_mock_vi.md"


class FakeEmbeddings:
    def embed_documents(self, texts):
        return [self._embed(text) for text in texts]

    def embed_query(self, text):
        return self._embed(text)

    def _embed(self, text):
        text = str(text).lower()
        return [
            float("giao" in text or "delivery" in text),
            float("trả" in text or "hoàn" in text),
            float("voucher" in text),
            float(len(text) % 997) / 997.0,
        ]


with TemporaryDirectory() as tmpdir:
    chroma_store = ChromaPolicyStore(Path(tmpdir), FakeEmbeddings())
    chroma_store.rebuild(POLICY_PATH)
    assert chroma_store.collection.count() == 38

    hits = chroma_store.search("thời gian giao hàng dự kiến", top_k=3)
    assert len(hits) == 3
    assert set(hits[0]) == {"citation", "content", "distance"}
    assert isinstance(hits[0]["content"], str)

    print("Collection count:", chroma_store.collection.count())
    pprint(hits)


Collection count: 38
[{'citation': '3. Thuật ngữ > 3.2. Giao hàng thành công',
  'content': '## 3. Thuật ngữ\n'
             '\n'
             '### 3.2. Giao hàng thành công\n'
             '\n'
             'Đơn được xem là giao hàng thành công khi hệ thống vận chuyển cập '
             'nhật trạng thái `delivered` hoặc tương đương, và hàng hóa đã tới '
             'đúng địa chỉ hoặc đúng người nhận được ủy quyền.',
  'distance': 0.019115090370178223},
 {'citation': '4. Chính sách giao hàng > 4.2. Thời gian xử lý đơn',
  'content': '## 4. Chính sách giao hàng\n'
             '\n'
             '### 4.2. Thời gian xử lý đơn\n'
             '\n'
             'Sau khi khách đặt hàng thành công, nhà bán hoặc kho vận hành sẽ '
             'cần một khoảng thời gian để:  \n'
             '- xác nhận tồn kho\n'
             '- đóng gói\n'
             '- in vận đơn\n'
             '- bàn giao cho đơn vị vận chuyển  \n'
             'Thời gian xử lý thông thường:  \n'
             '- đơn tiêu

## 12. Build Chroma index thật bằng OpenAI embeddings

Cell này dùng `OPENAI_API_KEY` trong `.env` và model embedding mặc định `text-embedding-3-small`.

In [54]:
from pathlib import Path
import importlib

from app.config import Settings
import rag.embeddings as embeddings_module
import rag.vector_store as vector_store_module

# Reload để notebook nhận code mới nhất sau khi sửa file .py.
importlib.reload(embeddings_module)
importlib.reload(vector_store_module)

from rag.embeddings import build_embedding_model
from rag.vector_store import ChromaPolicyStore

ROOT = Path.cwd()
POLICY_PATH = ROOT / "data" / "policy_mock_vi.md"

settings = Settings.load()
if not settings.openai_api_key:
    raise RuntimeError(
        "Missing OPENAI_API_KEY in .env. "
        "Hãy điền OPENAI_API_KEY=... rồi chạy lại cell này."
    )

embedding_model_name = settings.embedding_model_name
if not embedding_model_name.startswith(("text-embedding-", "openai:")):
    embedding_model_name = "text-embedding-3-small"

REAL_CHROMA_DIR = settings.chroma_dir
embedding_model = build_embedding_model(
    embedding_model_name,
    api_key=settings.openai_api_key,
)

real_policy_store = ChromaPolicyStore(REAL_CHROMA_DIR, embedding_model)
real_policy_store.rebuild(POLICY_PATH)

print("Persist dir:", REAL_CHROMA_DIR)
print("Embedding model:", embedding_model_name)
print("Collection count:", real_policy_store.collection.count())


Persist dir: /Users/thangnguyen/Documents/VinAI - Lab/Day09-MultiAgent-Architecture/src/.chroma
Embedding model: text-embedding-3-small
Collection count: 38


## 13. Search policy index thật

In [55]:
hits = real_policy_store.search("Đơn hàng 1971 có được hoàn trả không?", top_k=4)

for hit in hits:
    print("=" * 80)
    print("citation:", hit["citation"])
    print("distance:", hit["distance"])
    print(hit["content"][:800])

citation: 5. Chính sách đổi trả và hoàn tiền > 5.10. Quan hệ giữa trạng thái đơn hàng và quyền trả hàng
distance: 0.40922248363494873
## 5. Chính sách đổi trả và hoàn tiền

### 5.10. Quan hệ giữa trạng thái đơn hàng và quyền trả hàng

Trong dữ liệu lab, sinh viên nên hiểu rõ:  
- đơn chưa giao thành công thường **chưa thể** bắt đầu quy trình trả hàng thông thường
- đơn đang `in_transit` có thể được hỗ trợ hủy hoặc từ chối nhận trong một số trường hợp, nhưng không xem là "đã trả hàng"
- đơn `delivered` mới là trạng thái thường dùng để tính cửa sổ đổi trả
- đơn `completed` vẫn có thể còn trong hạn trả hàng nếu chưa vượt `eligible_for_return_until`  
Điều này rất quan trọng cho các câu hỏi như:  
- "Đơn hàng 1971 có được hoàn trả không?"
citation: 5. Chính sách đổi trả và hoàn tiền > 5.2. Các lý do được chấp nhận
distance: 0.4714675545692444
## 5. Chính sách đổi trả và hoàn tiền

### 5.2. Các lý do được chấp nhận

Yêu cầu sau bán có thể được xem xét khi thuộc một hoặc nhiều lý do sau:  
-

## 14. Setup `ShoppingAssistant` và supervisor-worker agents

Cell này khởi tạo node cha `ShoppingAssistant` và import các agent builder trong `app.nodes`.

In [ ]:
from app.graph import ShoppingAssistant
from app.nodes import (
    build_data_agent,
    build_policy_agent,
    build_response_agent,
    build_search_policy_tool,
    build_supervisor_graph,
    build_supervisor_prompt,
)

assistant = ShoppingAssistant()

print("Provider:", assistant.settings.provider)
print("LLM model:", assistant.settings.model)
print("Embedding model:", assistant.settings.embedding_model_name)
print("Embedding class:", type(assistant.embedding_model).__name__ if assistant.embedding_model else None)
print("Data tools:", [tool.name for tool in assistant.data_tools])

## 15. Test supervisor graph builder

Supervisor hiện là graph thật từ `langgraph_supervisor.create_supervisor`, không còn node route deterministic.

In [ ]:
supervisor_prompt = build_supervisor_prompt()
compiled_graph = build_supervisor_graph(assistant)

print("Compiled graph:", type(compiled_graph).__name__)
print("Prompt preview:")
print(supervisor_prompt[:1200])

assert "policy_worker" in supervisor_prompt
assert "data_worker" in supervisor_prompt
assert "response_worker" in supervisor_prompt
assert compiled_graph is not None

## 16. Test policy worker và `search_policy` tool

Policy worker là ReAct agent. Tool `search_policy` dùng Chroma + embedding thật; nếu thiếu `OPENAI_API_KEY`, cell sẽ báo lỗi rõ.

In [ ]:
policy_agent = build_policy_agent(assistant)
search_policy = build_search_policy_tool(assistant)

print("Policy agent:", type(policy_agent).__name__)
try:
    hits = search_policy.invoke({"query": "Voucher có được hoàn lại khi hủy đơn không?", "top_k": 4})
    for hit in hits:
        print("=" * 80)
        pprint(hit)
    assert hits
    assert set(hits[0]) == {"citation", "content", "distance"}
except RuntimeError as exc:
    print("search_policy chưa chạy được:", exc)
    print("Hãy set OPENAI_API_KEY trong .env nếu muốn test RAG thật.")

## 17. Test data worker và data tools

Data worker là ReAct agent dùng các tool nhỏ từ `data_access.py`. Cell này test trực tiếp từng tool nhỏ, không gọi LLM.

In [ ]:
data_agent = build_data_agent(assistant)
tools = {tool.name: tool for tool in assistant.data_tools}

order = tools["get_order_detail_by_order_id"].invoke({"order_id": "1971"})
missing_order = tools["get_order_detail_by_order_id"].invoke({"order_id": "9999"})
customer = tools["get_customer_by_id"].invoke({"customer_id": "C001"})
vouchers = tools["get_vouchers_by_customer_id"].invoke({"customer_id": "C001", "only_active": True})

print("Data agent:", type(data_agent).__name__)
pprint(order)
pprint(missing_order)
pprint(customer)
pprint(vouchers)

assert order["status"] == "ok"
assert missing_order["status"] == "not_found"
assert customer["status"] == "ok"
assert vouchers["status"] == "ok"

## 18. Test response worker builder

Response worker format câu trả lời cuối trong full graph. Cell này chỉ kiểm tra agent được build đúng; muốn gọi thật thì chạy cell `assistant.ask()` phía dưới.

In [ ]:
response_agent = build_response_agent(assistant)

print("Response agent:", type(response_agent).__name__)
assert response_agent is not None

## 19. Test node cha `assistant.ask()`

Cell này chạy workflow thật bằng `create_supervisor` + `create_react_agent`. Lưu ý: cell này gọi LLM provider thật và có thể tốn API/network.

In [ ]:
full_graph_questions = [
    "Chính sách hoàn trả hàng ra sao?",
    "Đơn hàng 1971 bao giờ được giao?",
    "Đơn hàng 1971 có được hoàn trả không?",
    "Voucher của tôi còn dùng được không?",
    "Kiểm tra đơn hàng 9999 giúp tôi",
]

for question in full_graph_questions:
    result = assistant.ask(question)
    print("=" * 80)
    print("Question:", question)
    print("Route:", result["route"])
    print("Final answer:")
    print(result["final_answer"][:1500])
    assert "route" in result
    assert "final_answer" in result
    assert "trace" in result

## 20. Inspect trace của một lần chạy graph

Trace của kiến trúc supervisor-worker là message history từ LangGraph, không còn list node deterministic.

In [ ]:
trace_result = assistant.ask("Đơn hàng 2058 còn trong thời gian trả hàng không?")
agent_names = [step.get("name") for step in trace_result["trace"] if step.get("name")]
print("Agent names:", agent_names)

assert trace_result["trace"]

for step in trace_result["trace"]:
    print("=" * 80)
    print("type:", step.get("type"), "name:", step.get("name"))
    pprint(step)


## 21. Test batch bằng `data/test.json`

Cell này chạy batch qua supervisor-agent thật. Kết quả route/status có thể nondeterministic vì phụ thuộc LLM; cell chỉ in mismatch để debug, không assert fail cứng.

In [ ]:
BATCH_TRACE_DIR = ROOT / "src" / "artifacts" / "notebook_traces"
summary = assistant.run_batch(ROOT / "data" / "test.json", BATCH_TRACE_DIR)

failures = []
for item in summary["results"]:
    route_ok = set(item["expected_route"]) == set(item["actual_route"])
    status_ok = item["expected_status"] == item["actual_status"]
    print(item["id"], "route=", item["actual_route"], route_ok, "status=", item["actual_status"], status_ok)
    if not (route_ok and status_ok):
        failures.append(item)

print("Failures:", len(failures))
print("Summary file:", BATCH_TRACE_DIR / "summary.json")